# Label_Generation_QC_v1

**Goal:** Generate and visually verify clean 4-class training labels from scratch \
before any patch generation or model training.

**Architecture:**
- Droplet detection via NPC watershed (histogram-clipped) with 10 px erosion boundary
- Nucleus detection via adaptive threshold with per-droplet Otsu fallback
- NPC puncta detection nucleus-boundary-relative (annular zone ± `NPC_MARGIN_PX`)
- All logic validated stage-by-stage before patch generation

| Dim order | `(T, Z, C, Y, X)` |
|---|---|
| Channels | `C=0 Membrane \| C=1 NLS \| C=2 NPC` |
| 4-class scheme | `0=Background \| 1=Droplet \| 2=NPC \| 3=Nucleus` |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path
from skimage import filters, morphology, segmentation, measure
from scipy import ndimage

In [ ]:
# ── Paths ───────────────────────────────────────────────────────────────────
HYPERSTACK_PATH = Path("/path/to/control_extract_1.1.tif")
OUTPUT_DIR      = Path("/path/to/patches")

# ── Channel indices ──────────────────────────────────────────────────────────
CH_MEMBRANE = 0
CH_NLS      = 1
CH_NPC      = 2

# ── 4-class label scheme ─────────────────────────────────────────────────────
CLASS_BACKGROUND = 0
CLASS_DROPLET    = 1
CLASS_NPC        = 2
CLASS_NUCLEUS    = 3

# ── Droplet detection — NPC watershed ────────────────────────────────────────
NPC_CLIP_PCT = 99.0  # histogram clip percentile to suppress NPC puncta signal
EROSION_PX   = 10   # pixels to erode from each watershed region edge

# ── NPC puncta detection — nucleus-boundary-relative ─────────────────────────
NPC_MARGIN_PX = 5   # annular half-width around nucleus edge (px, inward and outward)

# ── Nucleus detection — adaptive with Otsu fallback ──────────────────────────
ADAPTIVE_BLOCK_SIZE  = None  # None = auto-compute from droplet diameter (must be odd)
MAX_NUCLEUS_FRACTION = 0.80  # flag if nucleus mask > 80% of droplet area
MIN_NUCLEUS_FRACTION = 0.01  # flag if nucleus mask < 1% of droplet area (empty)

# ── Patch generation ──────────────────────────────────────────────────────────
PATCH_SIZE = 512
Z_FLOOR    = 6    # exclude coverslip artifact planes below this z-index
N_PREVIEW  = 10   # patches to spot-check in Stage 5

In [ ]:
def load_hyperstack(path: Path) -> np.ndarray:
    """
    Load the hyperstack TIFF and return as (T, Z, C, Y, X) array.

    Parameters
    ----------
    path : Path

    Returns
    -------
    np.ndarray, shape (T, Z, C, Y, X)
    """
    hs = tifffile.imread(str(path))
    print(f"Loaded: shape={hs.shape}  dtype={hs.dtype}")
    assert hs.ndim == 5, f"Expected 5D (T,Z,C,Y,X), got {hs.ndim}D"
    return hs


hyperstack = load_hyperstack(HYPERSTACK_PATH)
T, Z, C, Y, X = hyperstack.shape
print(f"T={T}  Z={Z}  C={C}  Y={Y}  X={X}")

---
## Stage 1 — Core Detection Functions & Single-Droplet Audit

Define the four core detection functions, then audit one droplet to confirm \
each class is assigned correctly **before** generating at scale.

| Function | Responsibility |
|---|---|
| `detect_droplets_npc_watershed` | NPC channel watershed → per-droplet masks |
| `detect_nucleus_adaptive` | Adaptive threshold + Otsu fallback → nucleus mask |
| `detect_npc_puncta` | Nucleus-boundary-relative → NPC puncta mask |
| `build_label_plane` | Compose 4-class label from per-droplet results |

In [ ]:
def detect_droplets_npc_watershed(
    npc_plane: np.ndarray,
    clip_pct: float = NPC_CLIP_PCT,
    erosion_px: int = EROSION_PX,
) -> list[dict]:
    """
    Detect individual droplets from a single NPC channel plane.

    Strategy
    --------
    1. Clip histogram at clip_pct to suppress bright NPC puncta signal.
       This is critical — without clipping, puncta create local maxima
       that break the watershed into fragments inside droplets.
    2. Gaussian blur on the clipped image to smooth before thresholding.
    3. Global threshold to obtain a binary foreground mask.
    4. Watershed to separate touching / merged droplets.
    5. Erode each watershed region by erosion_px to define the safe interior
       (prevents boundary bleed into neighbouring droplets; 10 px default).

    Parameters
    ----------
    npc_plane : np.ndarray, shape (Y, X)
        Single 2D plane from the NPC channel (C=2).
    clip_pct : float
        Percentile at which to clip the histogram before watershed.
        Default 99.0 — increase if puncta signal is still fragmenting droplets.
    erosion_px : int
        Pixels to erode from each watershed region edge. Default 10.

    Returns
    -------
    list of dict, one entry per detected droplet:
        {
            'label'    : int,         # watershed region label index
            'mask'     : np.ndarray,  # bool (Y, X), full-image eroded mask
            'bbox'     : tuple,       # (min_row, min_col, max_row, max_col)
            'centroid' : tuple,       # (row, col)
            'area'     : int,         # pixel count of eroded mask
        }
    """
    raise NotImplementedError(
        "TODO: clip histogram → Gaussian blur → threshold → watershed → "
        "erode each region by erosion_px → build and return droplet list"
    )

In [ ]:
def detect_nucleus_adaptive(
    nls_crop: np.ndarray,
    droplet_mask_crop: np.ndarray,
    block_size: int | None = ADAPTIVE_BLOCK_SIZE,
    fallback: str = "otsu",
) -> tuple[np.ndarray, str]:
    """
    Detect the nucleus interior from the NLS channel crop of a single droplet.

    Strategy
    --------
    Primary — adaptive (skimage.filters.threshold_local):
        Applied only within the droplet mask crop. block_size is auto-computed
        as ~1/3 of the droplet diameter if not supplied (must be odd).
        Result must pass both area-fraction gates below.

    Fallback — per-droplet Otsu (skimage.filters.threshold_otsu):
        Used when the adaptive result is empty or implausibly sized.
        Otsu computed on NLS signal masked to the droplet interior only.

    Validation gates — trigger fallback if either fails:
        nucleus_area < MIN_NUCLEUS_FRACTION * droplet_area  →  too small / empty
        nucleus_area > MAX_NUCLEUS_FRACTION * droplet_area  →  threshold too low

    Parameters
    ----------
    nls_crop : np.ndarray, shape (H, W)
        NLS channel (C=1) cropped to droplet bounding box.
    droplet_mask_crop : np.ndarray, shape (H, W), bool
        Droplet interior mask cropped to same bounding box.
    block_size : int or None
        Adaptive threshold block size. None = auto from droplet diameter.
    fallback : str
        Fallback strategy. Currently only 'otsu' supported.

    Returns
    -------
    nucleus_mask : np.ndarray, shape (H, W), bool
        Nucleus interior mask in crop coordinates.
    method_used : str
        'adaptive' or 'otsu' — which method produced the final mask.
    """
    raise NotImplementedError(
        "TODO: compute block_size if None → adaptive threshold within droplet mask "
        "→ validate area fractions → fallback to Otsu if needed "
        "→ return (nucleus_mask, method_used)"
    )

In [ ]:
def detect_npc_puncta(
    npc_crop: np.ndarray,
    nucleus_mask_crop: np.ndarray,
    margin_px: int = NPC_MARGIN_PX,
) -> np.ndarray:
    """
    Detect NPC puncta using the nucleus boundary as spatial anchor.

    Biological rationale
    --------------------
    NPC puncta sit on the nuclear envelope. The NLS signal maps perfectly
    to the internal surface of the NPC ring. Signal visible deeper inside
    the nucleus in the NPC channel is antibody diffusion artifact — excluded
    by construction. Signal at the droplet wall is the v6-era error — also
    excluded here because we anchor to the nucleus boundary, not the droplet wall.

    Strategy
    --------
    1. Compute nucleus boundary: morphological edge of nucleus_mask_crop
       (binary_dilation XOR mask, or morphological gradient).
    2. Build annular search zone:
           outer = binary_dilation(nucleus_mask, radius=margin_px)
           inner = binary_erosion(nucleus_mask,  radius=margin_px)
           search_zone = outer AND NOT inner
       This creates a thin ring centred on the nucleus edge.
    3. Threshold NPC signal within the search zone only.
       Threshold = mean + 2*std of NPC values within the full droplet interior
       (consistent with v7 calibration on control_extract_1.1.tif).
    4. Return binary puncta mask clipped to the search zone.

    Parameters
    ----------
    npc_crop : np.ndarray, shape (H, W)
        NPC channel (C=2) cropped to droplet bounding box.
    nucleus_mask_crop : np.ndarray, shape (H, W), bool
        Nucleus interior mask in crop coordinates (from detect_nucleus_adaptive).
    margin_px : int
        Half-width of the annular search zone around the nucleus edge. Default 5.

    Returns
    -------
    npc_mask : np.ndarray, shape (H, W), bool
        NPC puncta mask in crop coordinates.
    """
    raise NotImplementedError(
        "TODO: nucleus edge → outer dilation + inner erosion → annular search zone "
        "→ mean+2std threshold within zone → return npc_mask"
    )

In [ ]:
def build_label_plane(
    shape: tuple,
    droplets: list[dict],
    nucleus_masks: list[np.ndarray],
    npc_masks: list[np.ndarray],
) -> np.ndarray:
    """
    Compose the 4-class label plane from per-droplet segmentation results.

    Paint order (each step overwrites the previous):
        0  Background  →  default everywhere
        1  Droplet     →  eroded watershed interior for each droplet
        3  Nucleus     →  nucleus interior (overwrites droplet class)
        2  NPC         →  puncta on nuclear envelope (painted last — preserves
                          thin annular signal at the nucleus/droplet boundary)

    Note on NPC paint order: puncta sit at the nucleus boundary so minor
    spatial overlap with the nucleus mask is expected. NPC wins to preserve
    the thin annular class-2 signal.

    Parameters
    ----------
    shape : tuple (H, W)
        Output label plane shape (full image, not crop coordinates).
    droplets : list of dict
        Output of detect_droplets_npc_watershed. Provides 'mask' and 'bbox'.
    nucleus_masks : list of np.ndarray
        One nucleus mask per droplet, in crop coordinates. Same order as droplets.
    npc_masks : list of np.ndarray
        One NPC mask per droplet, in crop coordinates. Same order as droplets.

    Returns
    -------
    label_plane : np.ndarray, shape (H, W), dtype uint8
        4-class label image ready for patch extraction.
    """
    raise NotImplementedError(
        "TODO: init zeros (H,W) uint8 "
        "→ for each droplet: paint eroded mask→1, place nucleus_mask→3, place npc_mask→2 "
        "→ return label_plane"
    )

In [ ]:
def audit_single_droplet(
    hyperstack: np.ndarray,
    t: int,
    z: int,
    droplet_idx: int = 0,
) -> None:
    """
    STAGE 1 — Single-droplet label audit.

    Runs the full label pipeline on one droplet and renders a 2-row diagnostic:
        Row 1: Raw NLS | Raw NPC | Raw Membrane  (full plane, droplet highlighted)
        Row 2: Droplet mask | Nucleus mask | NPC mask | 4-class label  (crop)

    Use this to confirm class assignment is correct before generating at scale.
    No patches, no training — visual confirmation only.

    Parameters
    ----------
    hyperstack : np.ndarray, shape (T, Z, C, Y, X)
    t : int
        Timepoint index.
    z : int
        Z-plane index. Use a well-focused plane above Z_FLOOR.
    droplet_idx : int
        Which detected droplet to audit. 0 = largest by area (default).
    """
    raise NotImplementedError(
        "TODO: extract planes at (t, z) → detect_droplets_npc_watershed "
        "→ sort by area, pick droplet_idx → crop to bbox "
        "→ detect_nucleus_adaptive on NLS crop "
        "→ detect_npc_puncta on NPC crop "
        "→ build_label_plane "
        "→ matplotlib 2×4 subplot display"
    )

In [ ]:
# ── Stage 1 Runner ───────────────────────────────────────────────────────────
T_AUDIT   = 4   # mid-timecourse — pick a timepoint with visible nuclei
Z_AUDIT   = 8   # well-focused plane above Z_FLOOR
IDX_AUDIT = 0   # 0 = largest droplet by area

audit_single_droplet(hyperstack, t=T_AUDIT, z=Z_AUDIT, droplet_idx=IDX_AUDIT)

---
## Stage 2 — Watershed Separation Validation

Validate droplet separation at dense late timepoints (t=6–9). \
Each detected region should be approximately circular and individually separated. \
Merged or kidney-bean shaped regions indicate watershed failure.

In [ ]:
def validate_watershed(
    hyperstack: np.ndarray,
    t_range: list[int],
    z: int,
) -> None:
    """
    STAGE 2 — Watershed separation validation for dense timepoints.

    For each t in t_range, renders a 4-column diagnostic row:
        Col 1: Clipped NPC plane (what the watershed sees after histogram clip)
        Col 2: Watershed label map (coloured by region index)
        Col 3: Final eroded droplet masks overlaid on raw NPC
        Col 4: Detected droplet count (text annotation)

    Pass criterion: each coloured region ≈ circular and individually separated.
    Fail signal: merged blobs, concave or kidney-bean shaped regions.

    Parameters
    ----------
    hyperstack : np.ndarray, shape (T, Z, C, Y, X)
    t_range : list of int
        Timepoints to validate. Recommend [6, 7, 8, 9].
    z : int
        Z-plane index.
    """
    raise NotImplementedError(
        "TODO: for t in t_range: extract NPC plane "
        "→ detect_droplets_npc_watershed "
        "→ display 4-column diagnostic as one row per timepoint"
    )

In [ ]:
# ── Stage 2 Runner ───────────────────────────────────────────────────────────
T_RANGE_WATERSHED = list(range(6, 10))  # dense timepoints
Z_WATERSHED       = 8

validate_watershed(hyperstack, t_range=T_RANGE_WATERSHED, z=Z_WATERSHED)

---
## Stage 3 — NPC Puncta Label Verification

Confirm NPC labels lie on the nuclear envelope ring — not deep inside the nucleus \
(antibody diffusion artifact) and not at the droplet wall (v6-era error).

In [ ]:
def verify_npc_labels(
    hyperstack: np.ndarray,
    t: int,
    z: int,
    n_droplets: int = 5,
) -> None:
    """
    STAGE 3 — NPC puncta label verification.

    For n_droplets randomly sampled droplets, renders a 5-panel row:
        Panel 1: Raw NPC channel crop
        Panel 2: Nucleus mask (from NLS — the spatial anchor)
        Panel 3: Annular search zone (±NPC_MARGIN_PX around nucleus boundary)
        Panel 4: Final NPC puncta mask
        Panel 5: NPC mask (magenta) overlaid on NPC channel crop

    Red flags:
        Labels clustering at droplet wall  → spatial anchoring failure
        Labels flooding nucleus interior   → antibody artifact not excluded
        Empty NPC mask at visible NPC rings → threshold too high

    Parameters
    ----------
    hyperstack : np.ndarray, shape (T, Z, C, Y, X)
    t : int
        Timepoint with visible nuclei. Avoid t=0–1 (NLS signal too faint).
    z : int
        Z-plane index.
    n_droplets : int
        Number of droplets to randomly sample. Default 5.
    """
    raise NotImplementedError(
        "TODO: detect_droplets_npc_watershed "
        "→ random sample n_droplets from detected list "
        "→ for each: crop NLS + NPC → detect_nucleus_adaptive → detect_npc_puncta "
        "→ 5-panel subplot per droplet"
    )

In [ ]:
# ── Stage 3 Runner ───────────────────────────────────────────────────────────
T_NPC = 5   # timepoint with visible nuclei (avoid t=0–1)
Z_NPC = 8
N_NPC = 5   # droplets to sample

verify_npc_labels(hyperstack, t=T_NPC, z=Z_NPC, n_droplets=N_NPC)

---
## Stage 4 — Nucleus Threshold Sweep

Sweep adaptive + Otsu fallback across the full timecourse. \
Flag empty masks and implausibly large detections. \
Expected pattern: more Otsu fallbacks at early timepoints (faint NLS signal), \
larger stable nuclei at late timepoints.

In [ ]:
def nucleus_threshold_sweep(
    hyperstack: np.ndarray,
    t_range: list[int],
    z: int,
) -> None:
    """
    STAGE 4 — Nucleus threshold sweep across the timecourse.

    For each t in t_range, runs detect_nucleus_adaptive on all detected droplets.
    Records per-droplet: nucleus/droplet area fraction, method used, flagged status.

    Displays two outputs:
        1. Per-timepoint summary bar chart:
               Mean nucleus/droplet area fraction
               % using adaptive vs Otsu fallback
               % flagged (empty or implausibly large)
        2. Worst-case gallery: raw NLS crop + nucleus mask for every flagged droplet

    Parameters
    ----------
    hyperstack : np.ndarray, shape (T, Z, C, Y, X)
    t_range : list of int
        Timepoints to sweep. Recommend list(range(0, T)).
    z : int
        Z-plane index.
    """
    raise NotImplementedError(
        "TODO: for t in t_range: detect_droplets → for each droplet: "
        "crop NLS + mask → detect_nucleus_adaptive "
        "→ record (area_fraction, method, flagged) "
        "→ summary bar chart + flagged droplet gallery"
    )

In [ ]:
# ── Stage 4 Runner ───────────────────────────────────────────────────────────
T_RANGE_SWEEP = list(range(0, T))  # full timecourse
Z_SWEEP       = 8

nucleus_threshold_sweep(hyperstack, t_range=T_RANGE_SWEEP, z=Z_SWEEP)

---
## Stage 5 — Patch Generation with Inline QC

> ⚠️ **Only run after Stages 1–4 pass visual inspection.**

Generates patch pairs (3-channel input + 4-class label) for all valid (t, z) \
combinations and spot-checks a random sample before committing to a training run.

In [ ]:
def generate_patches_with_qc(
    hyperstack: np.ndarray,
    t_range: list[int],
    z_range: list[int],
    patch_size: int = PATCH_SIZE,
    output_dir: str = str(OUTPUT_DIR),
    n_preview: int = N_PREVIEW,
) -> None:
    """
    STAGE 5 — Patch generation with inline QC spot-check.

    For each (t, z) combination:
        1. detect_droplets_npc_watershed → per-droplet masks
        2. detect_nucleus_adaptive       → nucleus mask per droplet
        3. detect_npc_puncta             → NPC mask per droplet
        4. build_label_plane             → 4-class label (H, W)
        5. Extract patch_size × patch_size crops centred on each droplet centroid
        6. Save as .npy pairs: input (C, H, W) float32 + label (H, W) uint8

    After generation, display n_preview randomly sampled patch pairs:
        Row 1: NLS | NPC | Membrane  (raw channel crops)
        Row 2: 4-class label  (background=purple, droplet=teal, NPC=yellow, nucleus=green)

    Output naming convention:
        patch_t{t:02d}_z{z:02d}_d{droplet_idx:04d}.npy

    Parameters
    ----------
    hyperstack : np.ndarray, shape (T, Z, C, Y, X)
    t_range : list of int
    z_range : list of int
        Use list(range(Z_FLOOR, Z)) to exclude coverslip artifact planes.
    patch_size : int
        Patch edge length in pixels. Default 512.
    output_dir : str
        Directory to save .npy files. Created if not present.
    n_preview : int
        Number of random patches to display post-generation.
    """
    raise NotImplementedError(
        "TODO: Path(output_dir).mkdir → for (t, z): full pipeline "
        "→ for each droplet: pad-safe centred crop of input + label "
        "→ np.save → after loop: random sample n_preview → display 2-row grid"
    )

In [ ]:
# ── Stage 5 Runner ───────────────────────────────────────────────────────────
# Only run after Stages 1–4 pass visual inspection.
T_RANGE_PATCHES = list(range(0, T))
Z_RANGE_PATCHES = list(range(Z_FLOOR, Z))

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

generate_patches_with_qc(
    hyperstack,
    t_range    = T_RANGE_PATCHES,
    z_range    = Z_RANGE_PATCHES,
    patch_size = PATCH_SIZE,
    output_dir = str(OUTPUT_DIR),
    n_preview  = N_PREVIEW,
)